# More data

**Question:** does training on more sky (3 CCD quadrants of field 618 and a second field, 26 training units instead of 8)
make the model better? Read `docs/more_data.md` alongside this notebook.

Training takes ~35 minutes, so it lives in `scripts/train_more_data_bundle.py`; this notebook only evaluates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch

from asthunt.config.load_config import load_config
from asthunt.acquire.list_units import list_units
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.dataset.make_unit_rng import make_unit_rng
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.models.load_model_bundle import load_model_bundle
from asthunt.pipeline.run_night_pipeline import run_night_pipeline
from asthunt.evaluate.measure_depth_at_thresholds import measure_depth_at_thresholds

project = load_config("project"); paths = project["paths"]
configs = {name: load_config(name) for name in ["acquire", "inject", "detect", "link"]}
device = "mps" if torch.backends.mps.is_available() else "cpu"
for split in ["train", "validation", "test", "holdout"]:
    units = list_units(configs["acquire"], split)
    print(f"{split:10s} {len(units):2d} units, e.g. {make_unit_tag(units[-1])}")

## 1. The scoreboard: each model at its own chosen threshold

In [ ]:
board = pd.read_csv(paths["reports"] / "scoreboard.csv")
board = board[board["version"].isin(["v1", "v4", "v5"]) & (board["evaluated"] == board["evaluated"].max())]
board[["version", "split", "nights", "detection_ap", "bogus_kept_per_frame", "mag_at_50_completeness",
       "known_recovered", "unexplained_per_night"]].round(2)

On the test units v5 looks **shallower** (20.56 vs 21.34) but with **fewer alarms** (0.3 vs 5 per night).
Is v5 worse, or just stricter? Compare both models at the same false-alarm levels.

In [ ]:
sweeps = {}
for version in ["v4", "v5"]:
    bundle = load_model_bundle(paths["models"], version)
    results = [run_night_pipeline(unit, load_unit_metadata(unit, paths["raw"]), configs, bundle, paths,
                                  make_unit_rng(project["seed"], unit), device, with_injections=True, write_reports=False)
               for unit in list_units(configs["acquire"], "test")]
    thresholds = sorted({0.1, 0.3, 0.5, 0.7, bundle["thresholds"]["tracklet"], 0.9})
    sweeps[version] = measure_depth_at_thresholds(results, thresholds)
    print(version, "chosen threshold", round(bundle["thresholds"]["tracklet"], 2))
    print(sweeps[version].round(2).to_string(index=False))

In [ ]:
for version, table in sweeps.items():
    plt.plot(table["unexplained_per_night"], table["mag_at_50_completeness"], "o-", label=version)
plt.xlabel("unexplained alarms per night (what a human must vet)"); plt.ylabel("depth: 50% completeness (mag)")
plt.title("test units, same night, every threshold"); plt.legend(); plt.show()

The curves nearly overlap: **at equal alarms v4 is equal or slightly better** (~0.1 mag at 5+ alarms per night, ~0.25 at ~3).
Most of v5's apparent loss came from its chosen threshold (set for 95% recall on its validation tracklets), which landed at a
much stricter point on the curve.

## Exercises
1. Which point on the curve would you choose for a live system where a person vets every alert? For an automated one?
2. Why does more training data not make the single-frame pipeline deeper? (Hint: deep_search.)
3. The validation AP of v5's tracklet classifier is 0.997. Why doesn't that predict the test depth?